# OR-Tools Computation Library EDA

OR-Tools is useful for constrained target construction: selecting non-overlapping trades, allocating scarce labels, or solving discrete assignment problems. This notebook connects it to existing target-engineering ideas.

In [1]:
from __future__ import annotations

import importlib
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

def required_library_status(module_name: str) -> pd.DataFrame:
    try:
        module = importlib.import_module(module_name)
        return pd.DataFrame([{
            "module": module_name,
            "required": True,
            "installed": True,
            "version": getattr(module, "__version__", "unknown"),
            "module_file": getattr(module, "__file__", None),
        }])
    except Exception as exc:
        return pd.DataFrame([{
            "module": module_name,
            "required": True,
            "installed": False,
            "version": None,
            "module_file": None,
            "error": f"{type(exc).__name__}: {exc}",
        }])

def synthetic_prices(rows: int = 360, symbol: str = "AAPL") -> pd.DataFrame:
    rng = np.random.default_rng(7)
    dates = pd.date_range("2024-01-02", periods=rows, freq="B")
    drift = 0.00055
    shock = rng.normal(0, 0.015, rows)
    close = 100 * np.exp(np.cumsum(drift + shock))
    open_ = close * (1 + rng.normal(0, 0.003, rows))
    high = np.maximum(open_, close) * (1 + rng.uniform(0.001, 0.02, rows))
    low = np.minimum(open_, close) * (1 - rng.uniform(0.001, 0.02, rows))
    volume = rng.integers(900_000, 4_500_000, rows).astype(float)
    return pd.DataFrame({
        "symbol": symbol,
        "date": dates,
        "open": open_,
        "high": high,
        "low": low,
        "close": close,
        "volume": volume,
    })

prices = synthetic_prices()
prices.head()

,symbol,date,open,high,low,close,volume
0,AAPL,2024-01-02,100.2080,101.2355,98.8738,100.0569,"4,479,129.0000"
1,AAPL,2024-01-03,101.1259,102.2340,100.4497,100.5615,"2,365,734.0000"
2,AAPL,2024-01-04,100.3819,101.9042,99.3952,100.2040,"1,409,205.0000"
3,AAPL,2024-01-05,98.9452,99.8670,97.0239,98.9286,"4,225,356.0000"
4,AAPL,2024-01-08,97.8130,98.6722,97.4943,98.3103,"2,886,674.0000"


## Required Dependency Status

In [2]:
required_library_status("ortools")

,module,required,installed,version,module_file
0,ortools,True,True,9.15.6755,/home/jlee153232/miniconda3/lib/python3.13/sit...


## Target Engineering: Constrained Trade Selection

Existing target engineering already has oracle trade solvers. OR-Tools is the right place for future discrete constraints such as maximum active trades, sector caps, non-overlap windows, or cardinality limits.

In [3]:
from ortools.sat.python import cp_model

candidates = pd.DataFrame({
    "trade_id": ["T1", "T2", "T3", "T4", "T5", "T6"],
    "symbol": ["AAPL", "AAPL", "MSFT", "MSFT", "NVDA", "NVDA"],
    "start": [0, 3, 1, 8, 2, 10],
    "end": [6, 9, 5, 13, 7, 15],
    "expected_return_bp": [420, 260, 310, 180, 520, 230],
})

model = cp_model.CpModel()
select = {row.trade_id: model.NewBoolVar(row.trade_id) for row in candidates.itertuples()}

# At most one selected trade can be active on each synthetic day.
for day in range(int(candidates["end"].max()) + 1):
    active = [select[row.trade_id] for row in candidates.itertuples() if row.start <= day <= row.end]
    if active:
        model.Add(sum(active) <= 1)

# At most one selected trade per symbol in this toy label panel.
for symbol, group in candidates.groupby("symbol"):
    model.Add(sum(select[row.trade_id] for row in group.itertuples()) <= 1)

model.Maximize(sum(int(row.expected_return_bp) * select[row.trade_id] for row in candidates.itertuples()))
solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 5
status = solver.Solve(model)

selected = candidates[[solver.Value(select[row.trade_id]) == 1 for row in candidates.itertuples()]].copy()
selected, solver.ObjectiveValue(), solver.StatusName(status)

(  trade_id symbol  start  end  expected_return_bp
 3       T4   MSFT      8   13                 180
 4       T5   NVDA      2    7                 520,
 700.0,
 'OPTIMAL')

## Existing Quant Warehouse Target Labels

In [4]:
from quant_warehouse.target_engineering import generate_optimal_events, add_action_labels, add_binary_classification_labels

price_frame = prices.set_index("date")[["open", "high", "low", "close", "volume"]].rename(columns={"high": "adj_high", "low": "adj_low"})
price_frame["high"] = price_frame["adj_high"]
price_frame["low"] = price_frame["adj_low"]
events = generate_optimal_events(
    price_frame,
    {"ME": [2]},
    min_profit_pct=0.03,
    buy_execution="adj_high",
    sell_execution="adj_low",
    short_execution="adj_low",
    cover_execution="adj_high",
    price_col="close",
)
actions = add_action_labels(events)
binary = add_binary_classification_labels(events, use_sample_weight=True)
print(events.head().to_string())
print(actions.head().to_string())
print(binary.head().to_string())

            event   side horizon       trade_id entry_date  exit_date  entry_px  exit_px  trade_duration_days  trade_return
date                                                                                                                       
2024-01-03  entry  short   ME_k2  short:ME:k2:1 2024-01-03 2024-01-31  100.5615  89.2279                   28        0.1127
2024-01-31   exit  short   ME_k2  short:ME:k2:1 2024-01-03 2024-01-31  100.5615  89.2279                   28        0.1127
2024-02-05  entry  short   ME_k2  short:ME:k2:2 2024-02-05 2024-02-29   88.2570  79.5151                   24        0.0990
2024-02-29   exit  short   ME_k2  short:ME:k2:2 2024-02-05 2024-02-29   88.2570  79.5151                   24        0.0990
2024-03-05  entry   long   ME_k2   long:ME:k2:3 2024-03-05 2024-03-26   79.5125  85.9147                   21        0.0805
            label  market_position   side horizon  trade_return
date                                                           
2024

## Feature Engineering Connection

OR-Tools is generally not a feature generator. It can create target labels or constrained research artifacts that are later joined to feature panels.

In [5]:
label_rows = selected.assign(target_name="or_tools_constrained_trade_selection", target_value=1)
label_rows

,trade_id,symbol,start,end,expected_return_bp,target_name,target_value
3,T4,MSFT,8,13,180,or_tools_constrained_trade_selection,1
4,T5,NVDA,2,7,520,or_tools_constrained_trade_selection,1


## Notes

- Keep OR-Tools logic in target engineering or dataset construction, not data-provider code.
- Good production uses: non-overlapping oracle labels, option candidate selection with constraints, label balancing, and portfolio-constrained target panels.
- Store solver metadata with generated labels: objective, constraints, version, and infeasibility status.